# Topic 5: Watermarking & Late Data

> **Phase 6 → Week 10 → Topic 5**

---

## Event Time vs Processing Time

```
Event Time:       when the event ACTUALLY HAPPENED  (embedded in the data)
Processing Time:  when Spark SEES the event         (wall clock when it arrives)

Example: A mobile app logs a click at 10:00:00 (event time)
         Network lag → event arrives at Spark at 10:03:00 (processing time)
         The event is 3 minutes LATE.

Why it matters:
  A 10-minute window from 10:00–10:10 should include this event.
  If you use processing time, the event falls in the 10:00–10:10 processing window.
  If you use event time, the event correctly goes into the 10:00–10:10 event window.

Rule: Always aggregate by EVENT TIME, not processing time.
```

---

## Watermark — How Spark Handles Late Data

```
df.withWatermark("event_time", "10 minutes")

Watermark = max(event_time seen so far) - 10 minutes

Example:
  Batch 1: events up to 10:20 → watermark = 10:10
  Batch 2: events up to 10:30 → watermark = 10:20
  
  A late event with event_time = 10:08 arrives in Batch 2:
    → event_time (10:08) < watermark (10:20)
    → EVENT IS DROPPED (too late)

  A late event with event_time = 10:15 arrives in Batch 2:
    → event_time (10:15) >= watermark (10:20)? NO, 10:15 < 10:20
    → EVENT IS DROPPED

  A late event with event_time = 10:25 arrives in Batch 2:
    → event_time (10:25) >= watermark (10:20) ✓
    → EVENT IS INCLUDED

Watermark also tells Spark when a window is "done" → evict its state.
Without watermark: state never evicted → unbounded memory.
```

---

## Watermark + Output Mode

```
Watermark + update mode:
  Windows are updated as late data arrives.
  Windows are finalized (evicted) after watermark passes them.
  → Best for streaming to Kafka or Delta (only send changes)

Watermark + append mode:
  Windows appear in output ONLY AFTER they are finalized.
  No partial results — only final counts appear.
  → Best for writing to file sinks (complete and final rows only)
```

---

## Interview Cheat Sheet

**Q: What is a watermark in Structured Streaming?**
> A watermark is a threshold that tells Spark how late data can arrive and still be processed. `withWatermark("event_time", "10 minutes")` means: accept events up to 10 minutes late. The watermark advances as `max(event_time seen) - 10 minutes`. Events older than the watermark are dropped. The watermark also enables Spark to safely evict state for windows that are older than the watermark.

**Q: What is the difference between event time and processing time windows?**
> Event time windows group data by the timestamp embedded in the event (when it happened). Processing time windows group data by when Spark processes it. Event time is almost always correct — a purchase at 10:05 should be in the 10:00–10:10 window regardless of when it arrives. Processing time can be simpler but incorrect for out-of-order or late data.

**Q: Can you use append mode with watermarked aggregations?**
> Yes — with append mode, a window's result only appears in the output after the watermark passes the window's end time (meaning the window is "closed" — no more late data expected). This produces final, non-changing results. Without watermark, append mode is not allowed for aggregations.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = SparkSession.builder \
    .appName("Week10 - Watermarking") \
    .master("local[4]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Spark ready")

## Part 1: Watermark — Basic Setup

In [ ]:
# Rate source gives us a real timestamp column — perfect for event time demos
# timestamp = actual system time when the row was generated

watermarked_stream = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 10) \
    .load() \
    .withColumn("category", F.element_at(
        F.array(F.lit("electronics"), F.lit("clothing"), F.lit("food")),
        (F.col("value") % 3 + 1).cast("int"))) \
    .withColumn("amount", (F.col("value") % 200 + 10).cast("double")) \
    .withWatermark("timestamp", "5 seconds")  # accept events up to 5s late

# Aggregate by category — WITH watermark enables state eviction
agg = watermarked_stream.groupBy("category").agg(
    F.count("*").alias("event_count"),
    F.round(F.sum("amount"), 2).alias("total")
)

# update mode: only emit rows that changed
q = agg.writeStream \
    .format("memory") \
    .queryName("watermarked_agg") \
    .outputMode("update") \
    .trigger(processingTime="3 seconds") \
    .start()

time.sleep(15)
print("Watermarked aggregation (update mode):")
spark.sql("SELECT * FROM watermarked_agg ORDER BY category").show()

q.stop()

## Part 2: Simulating Late Data

In [ ]:
import os, shutil
from datetime import datetime, timedelta

LATE_IN = "/tmp/late_data_stream"
LATE_CKPT = "/tmp/late_ckpt"
for p in [LATE_IN, LATE_CKPT]:
    if os.path.exists(p): shutil.rmtree(p)
    os.makedirs(p)

now = datetime.now()

def ts(minutes_ago):
    return (now - timedelta(minutes=minutes_ago)).strftime("%Y-%m-%d %H:%M:%S")

# Batch 1: recent events (should be accepted)
batch1 = spark.createDataFrame([
    (ts(0),  "A", 100.0),   # on time
    (ts(1),  "B",  50.0),   # 1 min ago — within 5 min watermark
    (ts(2),  "A",  80.0),   # 2 min ago — within 5 min watermark
], ["event_time", "category", "amount"]) \
    .withColumn("event_time", F.to_timestamp("event_time"))
batch1.coalesce(1).write.json(f"{LATE_IN}/batch1")

# Batch 2: mix of on-time and very late events
batch2 = spark.createDataFrame([
    (ts(0),  "B", 200.0),   # on time
    (ts(10), "A",  30.0),   # 10 min ago — older than 5 min watermark → DROPPED
    (ts(3),  "C",  60.0),   # 3 min ago — within watermark
], ["event_time", "category", "amount"]) \
    .withColumn("event_time", F.to_timestamp("event_time"))
batch2.coalesce(1).write.json(f"{LATE_IN}/batch2")

event_schema = StructType([
    StructField("event_time", TimestampType()),
    StructField("category",   StringType()),
    StructField("amount",     DoubleType()),
])

late_stream = spark.readStream \
    .schema(event_schema) \
    .json(LATE_IN) \
    .withWatermark("event_time", "5 minutes")  # 5-minute tolerance

late_agg = late_stream.groupBy("category").agg(
    F.count("*").alias("cnt"),
    F.sum("amount").alias("total")
)

q_late = late_agg.writeStream \
    .format("memory") \
    .queryName("late_data_result") \
    .outputMode("complete") \
    .option("checkpointLocation", LATE_CKPT) \
    .trigger(once=True) \
    .start()

q_late.awaitTermination()
print("Result (10-min-late event for category A dropped):")
spark.sql("SELECT * FROM late_data_result ORDER BY category").show()
print("Expected: A=180 (100+80), B=250 (50+200), C=60")
print("(The 30.0 for A from 10 mins ago was dropped by watermark)")

## Part 3: Watermark Mechanics Explained

In [ ]:
print("""
Watermark Mechanics Step-by-Step:
════════════════════════════════════════════════════════════════

df.withWatermark("event_time", "10 minutes")

Batch 1 arrives:
  Events: event_time = [10:05, 10:07, 10:12]
  Max event_time seen = 10:12
  Watermark = 10:12 - 10 min = 10:02
  → All 3 events >= 10:02, all ACCEPTED

Batch 2 arrives:
  Events: event_time = [10:15, 10:08, 09:45]
  Watermark before batch = 10:02
  Max event_time seen = 10:15 → new watermark = 10:05
  
  10:15 >= 10:05 → ACCEPTED
  10:08 >= 10:05 → ACCEPTED  (3 min late but within watermark)
  09:45 < 10:05  → DROPPED   (20 min late, past watermark)

Batch 3 arrives:
  Watermark = 10:05 (or higher)
  Windows ending before 10:05 can now be evicted from state store.
  → Memory freed ✓

════════════════════════════════════════════════════════════════

Output mode behavior with watermark:

  update mode:
    → Emits partial window results as data arrives
    → Window state evicted after watermark passes it
    → Best for: Kafka output, Delta output (low latency)

  append mode:
    → Emits window results ONLY after window is closed
    → Window closed when watermark > window_end_time
    → Best for: file sinks (only write finalized results)

  complete mode:
    → Emits full result every trigger (NO state eviction)
    → Watermark has no effect on eviction in complete mode
    → Avoid for long-running jobs (growing state)
════════════════════════════════════════════════════════════════
""")

## Part 4: Watermarked Deduplication (Memory-Safe)

In [ ]:
# dropDuplicates without watermark: state grows forever (never evicted)
# dropDuplicates WITH watermark: keys older than watermark are evicted

stream4 = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 10) \
    .load() \
    .withColumn("event_id", (F.col("value") % 30).cast("string")) \
    .withColumn("event_time", F.col("timestamp"))

# Memory-safe deduplication: events older than 10 seconds are evicted from state
deduped_safe = stream4 \
    .withWatermark("event_time", "10 seconds") \
    .dropDuplicates(["event_id", "event_time"])  # watermark column must be in dedup keys

q_dedup = deduped_safe.writeStream \
    .format("memory") \
    .queryName("safe_dedup") \
    .outputMode("append") \
    .trigger(processingTime="3 seconds") \
    .start()

time.sleep(15)
count = spark.sql("SELECT count(*) FROM safe_dedup").collect()[0][0]
print(f"Rows after watermarked dedup: {count}")
print("State is bounded — old event_ids are evicted after 10 seconds")

q_dedup.stop()

## Exercises

1. Create a stream with a `withWatermark("timestamp", "2 minutes")`. Simulate a late event by writing a file with a timestamp 10 minutes in the past. Verify it is dropped by checking the result count.
2. What is the difference between setting watermark to `"0 seconds"` vs `"1 hour"`? What are the tradeoffs for each?
3. You have a pipeline writing to a Parquet file sink with `outputMode="append")` and a watermarked groupBy. Why don't you see any results for the first few minutes? What determines when results start appearing?
4. A customer pays at 10:00 AM from an offline device. The payment event arrives at the Spark streaming job at 10:47 AM. Your watermark is `"30 minutes"`. Is the event accepted or dropped? What is the watermark value when the event arrives if the latest event time seen is 10:50?
5. Explain why `withWatermark()` must come BEFORE `groupBy()` in your code.